##### collapse

In [1]:
import torchvision.models as models

model = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

In [2]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [3]:
from torchvision import datasets

train_data = datasets.CIFAR10(
    train=True,
    download=False,
    root="data",
    transform=train_transform
)

test_data = datasets.CIFAR10(
    train=False,
    download=False,
    root="data",
    transform=test_transform
)

In [4]:
import torch.nn as nn

model.fc = nn.Linear(512, 10)

### What fine-tuning actually means

So far, you did feature extraction:

- pretrained CNN backbone
- frozen weights
- trained only the classifier

Fine-tuning means:

- unfreezing some pretrained layers
- letting them adapt to your dataset
- while keeping most knowledge intact

You are refining, not relearning vision.

### Why NOT unfreeze everything immediately

If you unfreeze the entire ResNet at once:

- pretrained features get overwritten
- gradients explode or drift
- training becomes unstable
- performance can get worse

This is called catastrophic forgetting.

So we unfreeze gradually.

### Mental model of a CNN

<pre>
Early layers   → edges, colors, textures
Middle layers  → shapes, parts
Late layers    → task-specific concepts
Classifier     → ImageNet classes
</pre>

Key rule:

- early layers = general
- late layers = specific

So we fine-tune from the top downward.

### Strategy we will use

We follow this order:

1. Train classifier only (already done)

2. Unfreeze last block

3. Use lower learning rate

4. Keep early layers frozen

### Inspect ResNet structure (know what to unfreeze)

ResNet-18 has these main blocks:

<pre>
conv1
layer1
layer2
layer3
layer4
fc
</pre>


- layer4 is the last convolutional block
- closest to classifier
- most task-specific

So we unfreeze layer4 + fc.

### Unfreeze selected layers

In [5]:
for param in model.parameters():
    param.requires_grad = False

Now selectively unfreeze:

In [6]:
for param in model.layer4.parameters():
    param.requires_grad = True
    
for param in model.fc.parameters():
    param.requires_grad = True

What this does:

- early layers remain frozen
- high-level features adapt
- training stays stable

### Why learning rate MUST change

Pretrained weights are already good.

If you use a large learning rate:
- you destroy useful features

So we reduce it.

In [7]:
import torch

optimizer = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 0.0001},
    {"params": model.fc.parameters(), "lr":0.001}
])

Why two learning rates:

- classifier needs faster learning
- pretrained layers need gentle updates

This is critical

### Everything else stays the same

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    shuffle=True,
    batch_size=128,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_data,
    shuffle=False,
    batch_size=128,
    num_workers=4,
    pin_memory=True
)

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

In [10]:
loss_fn = nn.CrossEntropyLoss()

In [11]:
epochs = 5

for epoch in range(epochs):
    model.train()
    avg_loss = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        loss = loss_fn(preds, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        avg_loss += loss.item()
        
    avg_loss /= len(train_loader)    
    print(f"Epoch: {epoch} | Avg loss per epoch: {avg_loss:.4f}")

Epoch: 0 | Avg loss per epoch: 0.4318
Epoch: 1 | Avg loss per epoch: 0.1331
Epoch: 2 | Avg loss per epoch: 0.0331
Epoch: 3 | Avg loss per epoch: 0.0105
Epoch: 4 | Avg loss per epoch: 0.0063


In [12]:
total = 0
correct = 0

model.eval()

for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    
    preds = model(images)
    predicted = preds.argmax(dim=1)
    
    correct += (predicted == labels).sum().item()
    
    total += labels.size(0)
    
accuracy = correct/total
print(accuracy)

0.908


i saw 90% accuracy (10% increase)

### When to stop unfreezing

Stop when:

- validation accuracy plateaus
- training becomes unstable
- gains are marginal

Unfreezing everything is rarely needed unless:
- dataset is very large
- domain is very different (medical images, satellite, etc.)